# Active Share of Estonian II-pillar pension funds

Computes **Active Share** (Cremers & Petajisto 2009) for every actively managed
Estonian II-pillar pension fund from the **look-through dataset** in this repository: holdings as of **31.05.2026**, equities at security level,
bonds at issuer level, benchmarks = each manager's own index fund + Tuleva's bond fund TUK00.

The dataset is the finished product of Part 1 (parsing + look-through + matching;
see `README.md` and `sources.csv` for provenance). This notebook is Part 2 in
full: **only arithmetic** — no data processing, good data quality is assumed.

Set `BASE` below to the local folder or to the GitHub raw URL of the dataset.


In [1]:
import json, urllib.request
import pandas as pd

BASE = "."   # local; or "https://raw.githubusercontent.com/StenEhrlich/estonian-pension-lookthrough/main"

funds   = pd.read_csv(f"{BASE}/funds.csv", index_col="code")
sleeves = pd.read_csv(f"{BASE}/fund_sleeves.csv").pivot(index="code", columns="sleeve", values="weight_pct")
fh      = pd.read_csv(f"{BASE}/fund_holdings.csv")       # active funds' vectors
bh      = pd.read_csv(f"{BASE}/benchmark_holdings.csv")  # index funds + TUK00
len(fh), len(bh)


(84886, 22945)

## 1. The formula

$$\text{Active Share} = \tfrac{1}{2}\sum_i \lvert w_{fund,i} - w_{bench,i}\rvert$$

over the union of positions $i$, both sides summing to 100 %. 0 % = identical portfolios,
100 % = nothing in common.


In [2]:
def vec(df, code, vector):
    """One fund's holdings vector as a dict {position_key: weight_pct} (sums to 100)."""
    rows = df[(df["code"] == code) & (df["vector"] == vector)]
    return dict(zip(rows["key"], rows["weight_pct"]))

def active_share(fund, bench):
    """AS = 1/2 * sum |w_fund - w_bench| over the union of positions, in percent."""
    return 0.5 * sum(abs(fund.get(k, 0) - bench.get(k, 0)) for k in set(fund) | set(bench))

# hand-worked check: 60/40 vs 50/50 -> 0.5*(10+10) = 10
assert active_share({"A": 60, "B": 40}, {"A": 50, "B": 50}) == 10
assert active_share({"A": 100}, {"B": 100}) == 100
print("formula OK")


formula OK


## 2. Manager configuration

Fixed in the signed-off method specs (`*_method_spec.md`) — which funds, which benchmark,
and the two composite rules:

* `other_slice` — what benchmarks the fund's non-equity part: `"tuk00"` = bonds+PE+RE+gold
  slice goes to TUK00 (Luminor/LHV/Swedbank); `"equity_index"` = only bonds go to TUK00,
  the PE/RE/cash slice is benchmarked to the equity index (SEB spec §3.3).
* `cash_rule` — `"neutral"` = benchmark holds the same cash as the fund (contributes 0 pp);
  `"in_other"` = cash sits inside the SEB other-slice.
* SEB's bond comparison uses TUK00's name-keyed vector (`tuk00_variant` in `funds.csv`).


In [3]:
MANAGERS = {
    "SEB":      dict(funds=["SEK100", "SEK50", "SEK25", "SEK00"],
                     benchmark="SIK75", other_slice="equity_index", cash_rule="in_other", tuk00="bond_namekeyed"),
    "LHV":      dict(funds=["LLK50", "LXK75", "LMK25", "LXK00"],
                     benchmark="LIK75", other_slice="tuk00", cash_rule="neutral", tuk00="bond"),
    "Luminor":  dict(funds=["NPK75", "NPK50", "NPK25", "NPK00"],
                     benchmark="NIK100", other_slice="tuk00", cash_rule="neutral", tuk00="bond"),
    "Swedbank": dict(funds=["K1960", "K1970", "K1980", "K1990", "K2000", "KKONS"],
                     benchmark="Ki", other_slice="tuk00", cash_rule="neutral", tuk00="bond"),
}


## 3. Sleeve measures — equity AS and bond AS

Equity sleeve vs the manager's own index fund (pure stock selection); bond sleeve vs TUK00
at issuer level. Each vector is already renormalised to 100 % of its sleeve in the dataset.


## 4. Composite AS — whole fund vs a benchmark weighted by the fund's own allocation

The benchmark holds: the fund's equity weight in the index fund, the non-equity weight in
TUK00 (per `other_slice`), and cash per `cash_rule`. Positions no passive benchmark can hold
(PE, real estate, gold) appear only on the fund side, so each contributes ½ × its weight —
they are **active by construction**, never neutralised. `EQ:`/`BD:` prefixes keep the key
spaces apart so the decomposition by sleeve is a simple filter.


In [4]:
def composite(code, cfg, bench_eq, tuk00):
    s = sleeves.loc[code]
    F = {}                                       # the fund, as % of its total
    for k, w in vec(fh, code, "equity").items():  F["EQ:" + k] = w * s["eq"] / s["total"]
    for k, w in vec(fh, code, "bond").items():    F["BD:" + k] = w * s["bond"] / s["total"]
    for bucket in ("re", "pe", "gold"):
        if s[bucket]: F[bucket.upper()] = s[bucket] / s["total"] * 100
    if s["cash"]: F["CASH"] = s["cash"] / s["total"] * 100

    B = {}                                       # the composite benchmark
    if cfg["other_slice"] == "equity_index":     # SEB: PE/RE/cash slice -> equity index
        eq_slice, tuk_slice, bench_cash = (s["eq"] + s["re"] + s["pe"] + s["cash"]) / s["total"], s["bond"] / s["total"], 0.0
    else:                                        # others: bonds+PE+RE+gold -> TUK00, cash neutral
        eq_slice = s["eq"] / s["total"]
        tuk_slice = (s["bond"] + s["re"] + s["pe"] + s["gold"]) / s["total"]
        bench_cash = F.get("CASH", 0)
    for k, w in bench_eq.items():  B["EQ:" + k] = w * eq_slice
    for k, w in tuk00.items():     B["BD:" + k] = B.get("BD:" + k, 0) + w * tuk_slice
    if bench_cash: B["CASH"] = bench_cash
    return F, B

def decompose(F, B):
    """Composite AS split into per-sleeve contributions (pp)."""
    keys = set(F) | set(B)
    part = lambda match: 0.5 * sum(abs(F.get(k, 0) - B.get(k, 0)) for k in keys if match(k))
    return {"equity": part(lambda k: k.startswith("EQ:")), "bond": part(lambda k: k.startswith("BD:")),
            "re": part(lambda k: k == "RE"), "pe": part(lambda k: k == "PE"),
            "gold": part(lambda k: k == "GOLD"), "cash": part(lambda k: k == "CASH")}


## 5. Compute — all funds, all three measures


In [5]:
rows = []
for manager, cfg in MANAGERS.items():
    bench_eq = vec(bh, cfg["benchmark"], "equity")
    tuk00 = vec(bh, "TUK00", cfg["tuk00"])
    for code in cfg["funds"]:
        s = sleeves.loc[code]
        f_eq, f_bd = vec(fh, code, "equity"), vec(fh, code, "bond")
        F, B = composite(code, cfg, bench_eq, tuk00)
        rows.append({"code": code, "fund": funds.loc[code, "name"], "manager": manager,
                     "equity_AS": active_share(f_eq, bench_eq) if s["eq"] > 0 else None,
                     "bond_AS": active_share(f_bd, tuk00) if f_bd else None,
                     "composite_AS": active_share(F, B),
                     **{f"pp_{k}": v for k, v in decompose(F, B).items()}})
res = pd.DataFrame(rows).set_index("code").round(2)
res[["fund", "equity_AS", "bond_AS", "composite_AS"]]


,fund,equity_AS,bond_AS,composite_AS
code,,,,
SEK100,SEB Pensionifond 18+,34.46,99.99,38.81
SEK50,SEB Pensionifond 55+,34.45,81.50,42.54
SEK25,SEB Pensionifond 60+,18.39,89.77,53.10
SEK00,SEB Pensionifond 65+,27.94,58.59,55.87
LLK50,LHV Pensionifond Ettevõtlik,95.37,80.27,89.96
LXK75,LHV Pensionifond Julge,94.88,80.27,84.18
LMK25,LHV Pensionifond Tasakaalukas,99.36,80.27,80.53
LXK00,LHV Pensionifond Rahulik,100.00,80.27,76.87
NPK75,Luminor 16-50,14.25,98.18,18.39


## 6. Verification against the published reference results

`reference_results.json` holds the adversarially reviewed numbers the thesis reports.
Every value must match to 0.01 pp — if this cell raises, do not use the output.


In [6]:
if BASE.startswith("http"):
    ref = json.load(urllib.request.urlopen(f"{BASE}/reference_results.json"))
else:
    ref = json.load(open(f"{BASE}/reference_results.json"))
for code, r in ref.items():
    for m in ("equity_AS", "bond_AS", "composite_AS"):
        mine, want = res.loc[code, m], r[m]
        assert (want is None and pd.isna(mine)) or abs(mine - want) < 0.01, (code, m, mine, want)
print(f"all {len(ref)} funds match the reference results")


all 18 funds match the reference results


## 7. Thesis output

The composite AS is the **AS column of the thesis ESMA table** (`tab:esma`); the
decomposition (pp per sleeve) supports the results discussion. Decimal commas as in the thesis.


In [7]:
order = ["SEK100", "SEK50", "SEK25", "SEK00", "LXK75", "LLK50", "LMK25", "LXK00",
         "NPK75", "NPK50", "NPK25", "NPK00", "K1960", "K1970", "K1980", "K1990", "K2000", "KKONS"]
table = res.loc[order, ["fund", "equity_AS", "bond_AS", "composite_AS",
                        "pp_equity", "pp_bond", "pp_re", "pp_pe", "pp_gold"]]
table.to_csv("active_share_results.csv")
for code, r in table.iterrows():                       # LaTeX rows for tab:esma (AS column)
    print(f"{r['fund']:35s} & {r['composite_AS']:.1f} \\\\".replace(".", ","))
table


SEB Pensionifond 18+                & 38,8 \\
SEB Pensionifond 55+                & 42,5 \\
SEB Pensionifond 60+                & 53,1 \\
SEB Pensionifond 65+                & 55,9 \\
LHV Pensionifond Julge              & 84,2 \\
LHV Pensionifond Ettevõtlik         & 90,0 \\
LHV Pensionifond Tasakaalukas       & 80,5 \\
LHV Pensionifond Rahulik            & 76,9 \\
Luminor 16-50                       & 18,4 \\
Luminor 50-56                       & 29,7 \\
Luminor 56+                         & 49,0 \\
Luminor 61-65                       & 65,0 \\
Swedbank Pensionifond K1960         & 68,5 \\
Swedbank Pensionifond K1970         & 53,0 \\
Swedbank Pensionifond K1980         & 51,6 \\
Swedbank Pensionifond K1990         & 1,3 \\
Swedbank Pensionifond K2000         & 50,0 \\
Swedbank Pensionifond Konservatiivne & 76,3 \\


,fund,equity_AS,bond_AS,composite_AS,pp_equity,pp_bond,pp_re,pp_pe,pp_gold
code,,,,,,,,,
SEK100,SEB Pensionifond 18+,34.46,99.99,38.81,32.55,3.30,1.74,1.21,0.00
SEK50,SEB Pensionifond 55+,34.45,81.50,42.54,30.55,6.86,4.17,0.96,0.00
SEK25,SEB Pensionifond 60+,18.39,89.77,53.10,11.10,36.81,4.97,0.23,0.00
SEK00,SEB Pensionifond 65+,27.94,58.59,55.87,2.48,53.39,0.00,0.00,0.00
LXK75,LHV Pensionifond Julge,94.88,80.27,84.18,44.17,24.69,4.17,8.35,2.80
LLK50,LHV Pensionifond Ettevõtlik,95.37,80.27,89.96,30.08,33.00,7.20,16.81,2.87
LMK25,LHV Pensionifond Tasakaalukas,99.36,80.27,80.53,8.19,56.37,9.20,3.56,3.21
LXK00,LHV Pensionifond Rahulik,100.00,80.27,76.87,0.49,73.55,0.00,0.00,2.82
NPK75,Luminor 16-50,14.25,98.18,18.39,13.44,4.39,0.22,0.34,0.00
